# Question 24 - 68. Text Justification

Given an array of strings `words` and a width `maxWidth`, format the text such that each line has exactly `maxWidth` characters and is fully (left and right) justified.

You should pack your words in a greedy approach; that is, pack as many words as you can in each line. Pad extra spaces `' '` when necessary so that each line has exactly `maxWidth` characters.

Extra spaces between words should be distributed as evenly as possible. If the number of spaces on a line does not divide evenly between words, the empty slots on the left will be assigned more spaces than the slots on the right.

For the last line of text, it should be left-justified, and no extra space is inserted between words.

**Note:**

- A word is defined as a character sequence consisting of non-space characters only.
- Each word's length is guaranteed to be greater than `0` and not exceed `maxWidth`.
- The input array `words` contains at least one word.

**Example 1:**

    Input: words = ["This", "is", "an", "example", "of", "text", "justification."], maxWidth = 16
    Output:
    [
       "This    is    an",
       "example  of text",
       "justification.  "
    ]

**Example 2:**

    Input: words = ["What","must","be","acknowledgment","shall","be"], maxWidth = 16
    Output:
    [
      "What   must   be",
      "acknowledgment  ",
      "shall be        "
    ]
    Explanation: Note that the last line is "shall be    " instead of "shall     be", because the last line must be left-justified instead of fully-justified.
    Note that the second line is also left-justified because it contains only one word.

**Example 3:**

    Input: words = ["Science","is","what","we","understand","well","enough","to","explain","to","a","computer.","Art","is","everything","else","we","do"], maxWidth = 20
    Output:
    [
      "Science  is  what we",
      "understand      well",
      "enough to explain to",
      "a  computer.  Art is",
      "everything  else  we",
      "do                  "
    ]

**Constraints:**

- `1 <= words.length <= 300`
- `1 <= words[i].length <= 20`
- `words[i]` consists of only English letters and symbols.
- `1 <= maxWidth <= 100`
- `words[i].length <= maxWidth`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy line packing + share out the spaces with divmod

📘 **Revise first:** [Pattern H · simulate carefully](./0-concepts-array-and-string.ipynb) · [Part 5.4–5.5 · join, ljust](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (hand out spaces one at a time) | O(L · maxWidth) | O(maxWidth) |
| 2️⃣ Optimized (greedy pack + divmod) | O(S + L · maxWidth) | O(maxWidth) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

You're laying out text like a newspaper column: every line must be **exactly `maxWidth` characters**, flush on **both** the left and right edges.

```
maxWidth = 16
"This    is    an"
"example  of text"
"justification.  "
```

There's no clever trick here. It's about **following four rules exactly**:
1. **Fill each line greedily:** keep adding words while they fit, with at least one space between words.
2. **Spread the extra spaces** between the words as evenly as possible. If they don't divide evenly, the **leftmost** gaps get one extra.
3. **A line with only one word:** put the word on the left and pad the right with spaces.
4. **The last line:** single spaces between words, then pad the right.

**Everyday picture: sharing sweets among children in a row.** You have, say, 7 spare spaces and 3 gaps between words. Everyone gets `7 // 3 = 2`, and there's `7 % 3 = 1` left over, which goes to the first gap. Gaps: **3, 2, 2**. Python's `divmod(7, 3)` gives `(2, 1)` in one step.

### Step 0 · Clarify before coding

🗣️ *"Let me restate the rules so I don't miss any: greedy packing; extra spaces spread evenly with the left gaps getting more; single-word lines and the last line are left-justified and padded on the right. Every output line is exactly maxWidth."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Can a word be longer than `maxWidth`? | **No.** | Every word fits on a line by itself. |
| Uneven spaces go where? | **Left** gaps get the extra. | `divmod` handles this. |
| Last line? | Single spaces, then pad right. | A separate rule. |
| One-word line? | Pad right. | Also avoids dividing by zero gaps! |

**Plan:** split the work into two helpers: **"which words go on this line?"** and **"how do I lay out this line?"**.

### Step 1 · Brute force: the "obvious" way

**Plain-English idea:** pack the words greedily. For a normal line, start every gap at 1 space, then hand out the remaining spaces **one at a time**, going left to right and wrapping around, until the line is full.

🗣️ *"A natural first version distributes spaces one at a time round-robin from the left. It's correct, and 'left gaps get more' happens automatically. But it loops once per extra space, and it hides the simple arithmetic."*

In [1]:
class SolutionBrute:
    def fullJustify(self, words: list[str], maxWidth: int) -> list[str]:
        lines, i = [], 0
        while i < len(words):
            # greedy packing
            j, letters = i, 0
            while j < len(words) and letters + len(words[j]) + (j - i) <= maxWidth:
                letters += len(words[j])
                j += 1
            line_words = words[i:j]
            gaps = len(line_words) - 1
            if j == len(words) or gaps == 0:
                line = " ".join(line_words)
                lines.append(line + " " * (maxWidth - len(line)))
            else:
                spaces = [1] * gaps
                remaining = maxWidth - letters - gaps
                k = 0
                while remaining > 0:            # one space at a time, left to right, wrapping
                    spaces[k % gaps] += 1
                    remaining -= 1
                    k += 1
                line = line_words[0]
                for g, w in zip(spaces, line_words[1:]):
                    line += " " * g + w
                lines.append(line)
            i = j
        return lines

### Step 2 · Optimize: the "aha" moment

> 💡 **Aha:** sharing spaces out one at a time is just **division with a remainder**.
> For a line whose words have `letters` characters in total and `gaps = (words on the line − 1)`:
> - `spare = maxWidth − letters`
> - `base, extra = divmod(spare, gaps)`
> - The first `extra` gaps get `base + 1` spaces; the rest get `base`.

**The packing test, in plain words:** can I add the next word? Count the letters I already have, plus the new word, plus **one space for each word already on the line** (one space before each new word). If that's ≤ `maxWidth`, it fits:

`letters + len(next_word) + (number of words already on the line) ≤ maxWidth`

🗣️ *"I process one line at a time. First I greedily take words while the letters plus one space per gap fit. Then I lay out the line: if it's the last line or has a single word, I join with single spaces and pad on the right. Otherwise I divide the spare spaces by the number of gaps with divmod: every gap gets the base amount and the first 'extra' gaps get one more. Each character is written once, so it's linear in the input plus output."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| Hand out spaces one at a time | ⚠️ | Correct, but slower and hides the rule. |
| **Greedy packing + `divmod`** | ✅ **best** | The problem **requires** greedy packing, and divmod is exactly "spread evenly, leftmost get extra". |
| "Prettiest" layout via dynamic programming (as in word processors) | ❌ | That's a **different** problem. This one demands greedy, so DP would give different (wrong) answers. |
| Build lines with `+=` | ⚠️ | Fine at this size, but a list + `"".join` is the habit to show. |
| `str.ljust(maxWidth)` | ✅ | A clean way to pad the last line and single-word lines. |

In [2]:
class Solution:
    def fullJustify(self, words: list[str], maxWidth: int) -> list[str]:
        result = []
        i, n = 0, len(words)
        while i < n:
            # 1) which words go on this line? (greedy)
            j, letters = i, 0
            while j < n and letters + len(words[j]) + (j - i) <= maxWidth:
                letters += len(words[j])
                j += 1
            line_words = words[i:j]
            gaps = j - i - 1

            # 2) how do I lay out this line?
            if j == n or gaps == 0:
                # last line, or a single word: left-justify and pad the right
                result.append(" ".join(line_words).ljust(maxWidth))
            else:
                base, extra = divmod(maxWidth - letters, gaps)
                parts = []
                for k, w in enumerate(line_words[:-1]):
                    parts.append(w)
                    parts.append(" " * (base + (1 if k < extra else 0)))   # leftmost gaps get +1
                parts.append(line_words[-1])
                result.append("".join(parts))
            i = j
        return result

### Step 3 · Toy example walkthrough (tell it like a story)

`words = ["This", "is", "an", "example", "of", "text", "justification."]`, `maxWidth = 16`

**Line 1, packing:**

| try word | letters so far | letters + word + spaces needed | fits in 16? |
|---|---|---|---|
| This (4) | 0 | 0 + 4 + 0 = 4 | ✅ |
| is (2) | 4 | 4 + 2 + 1 = 7 | ✅ |
| an (2) | 6 | 6 + 2 + 2 = 10 | ✅ |
| example (7) | 8 | 8 + 7 + 3 = 18 | ❌ stop |

Line words: `This is an`, letters = 8, gaps = 2.
`divmod(16 − 8, 2) = (4, 0)` → each gap gets 4 → `"This····is····an"` ✅

**Line 2:** `example of text` → letters = 13, gaps = 2.
`divmod(16 − 13, 2) = (1, 1)` → the first gap gets **2**, the second gets 1 → `"example··of·text"` ✅

**Line 3:** `justification.` is the last line → left-justify, pad → `"justification.··"` ✅

**Narrated:** "Line one fits This, is, an, which is 8 letters, so 8 spare spaces shared between 2 gaps: 4 each. Line two has 13 letters and 3 spare spaces between 2 gaps: one each, and the leftover one goes to the left gap. The last line is just left-aligned and padded."

In [3]:
cases = [
    (["This", "is", "an", "example", "of", "text", "justification."], 16,
     ["This    is    an", "example  of text", "justification.  "]),
    (["What", "must", "be", "acknowledgment", "shall", "be"], 16,
     ["What   must   be", "acknowledgment  ", "shall be        "]),
    (["Science", "is", "what", "we", "understand", "well", "enough", "to", "explain",
      "to", "a", "computer.", "Art", "is", "everything", "else", "we", "do"], 20,
     ["Science  is  what we", "understand      well", "enough to explain to",
      "a  computer.  Art is", "everything  else  we", "do                  "]),
    (["a"], 1, ["a"]),
    (["a", "b", "c"], 1, ["a", "b", "c"]),
]
for words, w, expected in cases:
    assert SolutionBrute().fullJustify(words, w) == expected
    got = Solution().fullJustify(words, w)
    assert got == expected and all(len(line) == w for line in got)
    for line in got:
        print(f"|{line}|")
    print()
print("All tests passed ✅")

|This    is    an|
|example  of text|
|justification.  |

|What   must   be|
|acknowledgment  |
|shall be        |

|Science  is  what we|
|understand      well|
|enough to explain to|
|a  computer.  Art is|
|everything  else  we|
|do                  |

|a|

|a|
|b|
|c|

All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | Time | Extra memory | In plain words |
|---|---|---|---|
| One space at a time | O(L · maxWidth) | O(maxWidth) | Loops once per spare space on each line. |
| **Greedy + divmod** | **O(S + L · maxWidth)** | **O(maxWidth)** | Each word is examined once; each output character is written once. |

`S` = total letters in all words, `L` = number of output lines.

**Why this time?** Packing looks at each word once overall (`i` jumps straight to `j`). Building each line writes exactly `maxWidth` characters. So the work is "input size + output size", the least any solution can do.

**Why this memory?** Besides the output, we hold the pieces of one line at a time.

**Rules checklist (say these out loud; they're where people slip):**
1. The packing test counts **one space per word already on the line**.
2. The **last line** uses single spaces and is padded on the right.
3. A **one-word line** is padded on the right (and this avoids dividing by zero gaps).
4. Uneven spaces go to the **left** gaps first.

**Edge cases to mention:** a word exactly `maxWidth` long, every word on its own line, a single word overall.

---

#### 🗣️ Full interview script (about 60 seconds)

*"This is a careful simulation, one line at a time, in two steps.*

*First, packing: I greedily add words while the total letters plus one space per gap still fit in maxWidth.*

*Second, layout: if it's the last line or the line has a single word, I join with single spaces and pad on the right. Otherwise I take the spare spaces, maxWidth minus letters, and divmod them by the number of gaps. Every gap gets the base amount, and the first 'extra' gaps get one more, so the left side gets the extra spaces as required.*

*Each word is examined once and each output character is written once, so it's linear in the input plus the output."*